In [ ]:
from keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from project_brain_decoder.config import get_project_root
from project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import numpy as np
import gc

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
folder = get_project_root() / "data" / "raw"
files = list(folder.glob("*.nwb"))
batch_size, window_size, input_dim = 128, 30, 192

In [ ]:
train = files[:187] # 60%
val = files[187:249] # 20%
test = files[249:] # 20%

In [ ]:
neural_list = []
targets_list = []
neural_val_list = []
targets_val_list = []
neural_test_list = []
targets_test_list = []
X_list = []
y_list = []
X_val_list = []
y_val_list = []
X_test_list = []
y_test_list = []

In [ ]:
# session = load_nwb(file_path=train[0])["target_mrs_velocity"]
# session.shape

In [ ]:
def make_windows(neural: np.array, # shape(T, C) - time * channels
                 targets: np.array, # shape(T,) or (T, out_dim)
                 window_size: int,
                 stride: int=1) -> tuple[np.array, np.array]:
    """Slice into (window_size, C) windows; targets aligned to last timestep of each window"""
    T, C = neural.shape
    X = np.lib.stride_tricks.sliding_window_view(neural, window_size, axis=0)[::stride] # (n_windows, C, window_size)
    X = X.transpose(0, 2, 1) # (n_windows, window_size, C)
    # target for each window = value at the end of the window
    y = targets[window_size - 1 :: stride][:X.shape[0]]
    return X, y

In [ ]:
# Train set
for file in train:
    session = load_nwb(file_path=file)
    spiking_band = session["neural_spiking_band"]
    threshold_crossings = session["neural_threshold_crossings"]
    index_velocity = session["target_index_velocity"]
    mrs_velocity = session["target_mrs_velocity"]
    neural = np.concatenate([spiking_band, threshold_crossings], axis=1) # 192 columns
    targets = np.column_stack([index_velocity, mrs_velocity]) # 2 columns
    neural_list.append(neural)
    targets_list.append(targets)

# del X_list, y_list
# gc.collect()

In [ ]:
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()
neural_scaler.fit(np.concatenate(neural_list))
targets_scaler.fit(np.concatenate(targets_list))

In [ ]:
for neural, targets in zip(neural_list, targets_list):
    X_scaled = neural_scaler.transform(neural_list)
    y_scaled = targets_scaler.transform(targets_list)
    X_w, y_w = make_windows(neural=X_scaled, targets=y_scaled, window_size=30, stride=1)
    X_list.append(X_w)
    y_list.append(y_w)

In [ ]:
X_train = np.concatenate(X_list, axis=0) # stacking sessions vertically
y_train = np.concatenate(y_list, axis=0)

In [ ]:
# Validation set
for file in val:
    session = load_nwb(file_path=file)
    spiking_band = session["neural_spiking_band"]
    threshold_crossings = session["neural_threshold_crossings"]
    index_velocity = session["target_index_velocity"]
    mrs_velocity = session["target_mrs_velocity"]
    neural = np.concatenate([spiking_band, threshold_crossings], axis=1) # 192 columns
    targets = np.column_stack([index_velocity, mrs_velocity]) # 2 columns
    neural_val_list.append(neural)
    targets_val_list.append(targets)

# del X_val_list, y_val_list
# gc.collect()

In [ ]:
for neural, targets in zip(neural_val_list, targets_val_list):
    scaled_val_n = neural_scaler.transform(neural)
    scaled_val_t = targets_scaler.transform(targets)
    X_val_win, y_val_win = make_windows(neural=scaled_val_n, targets=scaled_val_t, window_size=30, stride=1)
    X_val_list.append(X_val_win)
    y_val_list.append(y_val_win)

In [ ]:
X_val = np.concatenate(X_val_list)
y_val = np.concatenate(y_val_list)

In [ ]:
# Test set
for file in test:
    session = load_nwb(file_path=file)
    spiking_band = session["neural_spiking_band"]
    threshold_crossings = session["neural_threshold_crossings"]
    index_velocity = session["target_index_velocity"]
    mrs_velocity = session["target_mrs_velocity"]
    neural = np.concatenate([spiking_band, threshold_crossings], axis=1) # 192 columns
    targets = np.column_stack([index_velocity, mrs_velocity]) # 2 columns
    neural_test_list.append(neural)
    targets_test_list.append(targets)

# del X_test_list, y_test_list
# gc.collect()

In [ ]:
for neural, targets in zip(neural_test_list, targets_test_list):
    scaled_test_n = neural_scaler.transform(neural)
    scaled_test_t = neural_scaler.transform(targets_test_list)
    X_test_win, y_test_win = make_windows(neural=scaled_test_n, targets=scaled_test_t, window_size=30, stride=1)
    X_test_list.append(X_test_win)
    y_test_list.append(y_test_win)

In [ ]:
X_test = np.concatenate(X_test_list)
y_test = np.concatenate(y_test_list)

In [ ]:
def get_transformer(window_size, input_dim):
    input_layer = Input(shape=(window_size, input_dim))
    attention_1 = MultiHeadAttention(num_heads=4, key_dim=48)(input_layer, input_layer)
    attention_1 = Dropout(0.1)(attention_1)
    attention_1 = LayerNormalization()(attention_1 + input_layer) # skip connection
    # Feed forward block
    dense_1 = Dense(units=384, activation="relu")(attention_1)
    dense_2 = Dense(units=192)(dense_1)
    attention_2 = Dropout(0.1)(dense_2)
    attention_2 = LayerNormalization()(attention_1 + attention_2) # skip connection 2
    avg_pool = GlobalAveragePooling1D()(attention_2)
    output = Dense(units=2)(avg_pool)
    model = Model(inputs=[input_layer], outputs=[output])
    model.compile(optimizer=Adam(learning_rate=0.0005), loss="mse")
    return model

In [ ]:
def main(model):
    get_transformer(window_size, input_dim).fit(X_train, y_train, batch_size=batch_size, epochs=1, validation_data=(X_val, y_val), callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])